# Day 11 · Transfer Learning & Data Augmentation

**Objective:** Freeze pretrained ResNet18, replace its final layer, apply `torchvision.transforms.v2`, and train a Tomato Leaf Disease Classifier.

## 1. Download the dataset

We select only tomato disease folders from the Kaggle dataset.

In [ ]:
!pip -q install kaggle
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!mkdir -p /content/plant_data
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plant_data --unzip

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models
from torchvision.transforms import v2

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_dirs = list(Path('/content/plant_data').rglob('train'))
valid_dirs = list(Path('/content/plant_data').rglob('valid'))
if not train_dirs:
    raise FileNotFoundError("No 'train' folder found after extraction.")
train_dir = train_dirs[0]
valid_dir = valid_dirs[0] if valid_dirs else None

## 2. Select tomato classes and apply transforms.v2

Training augmentation introduces small variations such as flips and rotations. Validation images are only resized and normalized.

In [ ]:
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = v2.Compose([
    v2.Resize((224, 224)), v2.RandomHorizontalFlip(), v2.RandomRotation(15),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean, std)
])
valid_tf = v2.Compose([
    v2.Resize((224, 224)), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean, std)
])

train_all = datasets.ImageFolder(train_dir, transform=train_tf)
tomato_classes = [c for c in train_all.classes if c.startswith('Tomato___')]
class_to_new = {name: i for i, name in enumerate(tomato_classes)}

class TomatoSubset(Dataset):
    def __init__(self, source, class_names):
        self.source = source
        self.indices = [i for i, (_, label) in enumerate(source.samples)
                        if source.classes[label] in class_names]
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        image, old_label = self.source[self.indices[idx]]
        return image, class_to_new[self.source.classes[old_label]]

train_data = TomatoSubset(train_all, tomato_classes)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

valid_loader = None
if valid_dir:
    valid_all = datasets.ImageFolder(valid_dir, transform=valid_tf)
    valid_data = TomatoSubset(valid_all, tomato_classes)
    valid_loader = DataLoader(valid_data, batch_size=32)

print('Tomato classes:', tomato_classes)
print('Training images:', len(train_data))

## 3. Load ResNet18 and replace the final layer

The pretrained network already contains useful visual features. Freeze its parameters and train only a new final layer for the tomato classes.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in model.parameters():
    parameter.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, len(tomato_classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

## 4. Train and evaluate

In [ ]:
for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    print(f'Epoch {epoch+1}/5 | Loss: {total_loss/total:.4f} | Accuracy: {correct/total:.4f}')

if valid_loader:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            outputs = model(images.to(device))
            correct += (outputs.argmax(1).cpu() == labels).sum().item()
            total += labels.size(0)
    print(f'Validation accuracy: {correct/total:.4f}')

### Day 11 · Key takeaway

Transfer learning reuses visual features learned by a pretrained model. Freezing ResNet18 keeps its feature extractor fixed while the replacement classifier learns the tomato disease classes.